# YouTube API Scraper — Colab Runner

Runs `youtube_api_scraper.py` on Google Colab. It will:
1. Clone the repo (or pull latest if already cloned)
2. Install dependencies
3. Mount Google Drive so checkpoints survive Colab disconnects
4. Set your YouTube API keys (auto-rotates across all 3)
5. Run the scraper end-to-end

**Why Colab works where the cloud sandbox didn't:** Colab gives you a real consumer-grade IP, so `youtube-transcript-api` won't get blocked the way AWS/GCP datacenter IPs are.

---
**Total time:** ~6–8 hours per full key (10k quota/key). Keep the tab open or use Colab Pro for background execution.

## 1. Clone repo + install deps

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/deepak-glitch/ytllm.git"
REPO_DIR = "/content/ytllm"
BRANCH   = "main"  # change if you want to run a feature branch

if not os.path.isdir(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull origin $BRANCH

%cd $REPO_DIR

!pip install -q google-api-python-client youtube-transcript-api tqdm python-dotenv

## 2. Mount Google Drive (so checkpoints persist across reconnects)

Colab VMs reset after ~12h idle. Mounting Drive means we never lose work — the scraper just resumes from `checkpoint/` next time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Persistent dirs on Drive
import os, shutil
DRIVE_ROOT = "/content/drive/MyDrive/ytllm_scrape"
os.makedirs(DRIVE_ROOT, exist_ok=True)
for d in ["checkpoint"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)

# Symlink so the scraper writes/reads from Drive directly
for d in ["checkpoint"]:
    if os.path.exists(d) and not os.path.islink(d):
        # merge any existing local files into Drive first
        for f in os.listdir(d):
            src = os.path.join(d, f)
            dst = os.path.join(DRIVE_ROOT, d, f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
        shutil.rmtree(d)
    if not os.path.islink(d):
        os.symlink(f"{DRIVE_ROOT}/{d}", d)

# Also persist quota_state.json + channel_cache.json
for f in ["quota_state.json", "channel_cache.json"]:
    drive_path = f"{DRIVE_ROOT}/{f}"
    if os.path.exists(f) and not os.path.islink(f):
        if not os.path.exists(drive_path):
            shutil.copy2(f, drive_path)
        os.remove(f)
    if not os.path.islink(f):
        # create empty file in Drive if it doesn't exist, then symlink
        if not os.path.exists(drive_path):
            open(drive_path, 'w').close()
        os.symlink(drive_path, f)

!ls -la checkpoint quota_state.json channel_cache.json 2>/dev/null | head

## 3. Set API keys

**Recommended:** put them in Colab Secrets (left sidebar 🔑 icon) under name `YOUTUBE_API_KEYS` as `key1,key2,key3` — then this cell loads them automatically.

Otherwise paste below.

In [ ]:
import os

# Try Colab secrets first
try:
    from google.colab import userdata
    keys = userdata.get('YOUTUBE_API_KEYS')
    if keys:
        os.environ['YOUTUBE_API_KEYS'] = keys
        print(f"✅ Loaded {len(keys.split(','))} keys from Colab secrets")
except Exception:
    pass

# Fallback: paste here (or leave blank if .env was committed and pulled)
if not os.environ.get('YOUTUBE_API_KEYS'):
    os.environ['YOUTUBE_API_KEYS'] = (
        "PASTE_KEY_1,PASTE_KEY_2,PASTE_KEY_3"
    )

# Sanity check
n = len([k for k in os.environ['YOUTUBE_API_KEYS'].split(',') if k.startswith('AIza')])
print(f"Using {n} API key(s) — ~{n*10000} quota units/day total")

## 4. Run the scraper

This runs in the foreground so you can watch progress. Safe to interrupt — it resumes from `checkpoint/` on the next run.

In [ ]:
!python youtube_api_scraper.py

## 5. Copy outputs to Drive

Once it finishes, save the training JSONLs to Drive.

In [ ]:
import shutil, os
DRIVE_OUT = "/content/drive/MyDrive/ytllm_scrape/outputs"
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in ["everything_raw.jsonl", "everything_training.jsonl", "training_data_v8.jsonl"]:
    if os.path.exists(f):
        shutil.copy2(f, f"{DRIVE_OUT}/{f}")
        print(f"📦 {f} → {DRIVE_OUT}/{f}")
!ls -lh {DRIVE_OUT}